# Генерация синтетических данных и каскадная валидация

**Пайплайн:**
1. Загрузка и анализ данных
2. Генерация новых примеров через LLM (малые <25 и средние 25–49 блоки)
3. Каскадная валидация: перплексия → дедупликация → классификатор
4. Сборка финального датасета

**Промежуточные результаты сохраняются в папку `checkpoints/`** — если ядро упадёт, каждый шаг можно перезапустить отдельно.

## Colab: подключение Google Drive и установка зависимостей

Структура папки на Google Drive:
```
MyDrive/
└── VKR/
    └── code/
        ├── augmentation/
        │   ├── augmentation.ipynb
        │   └── requirements.txt
        └── Data/
            └── data.json
```


In [1]:
from google.colab import drive
drive.mount("/content/drive")

# Путь к папке проекта на Google Drive — измени если структура другая
DRIVE_PROJECT = "/content/drive/MyDrive/VKR/code"

# Путь к файлу с данными
DATA_PATH = f"{DRIVE_PROJECT}/data/data_after_eda.csv"

# Устанавливаем зависимости из requirements.txt
import subprocess
result = subprocess.run(
    ["pip", "install", "-q", "-r", f"{DRIVE_PROJECT}/augmentation/requirements.txt"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Зависимости установлены успешно")
else:
    print("Ошибка установки:")
    print(result.stderr)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Зависимости установлены успешно


In [2]:
# ============================================================
# КОНФИГУРАЦИЯ — меняй здесь
# ============================================================


TARGET_COL = "label"                  # Колонка с метками классов
TEXT_COL = "text"                     # Колонка с текстами

SMALL_THRESHOLD = 25                  # Малые блоки: < 25 примеров
MEDIUM_THRESHOLD = 50                 # Средние блоки: 25–49 примеров
TARGET_COUNT = 50                     # До скольких примеров дополнять

# Модель для генерации
GENERATOR_MODEL = "ai-forever/rugpt3large_based_on_gpt2"

# Модель для фильтра перплексии
PERPLEXITY_MODEL = "ai-forever/rugpt3large_based_on_gpt2"
MAX_PERPLEXITY = 150.0

# Модель для эмбеддингов (дедупликация)
EMBED_MODEL = "cointegrated/rubert-tiny2"
SIMILARITY_THRESHOLD = 0.9

# Классификатор (укажи путь если есть дообученный)
CLASSIFIER_MODEL = None               # Например: "./my_classifier"
CLASSIFIER_THRESHOLD = 0.5

print("Конфигурация загружена")
print(f"Данные: {DATA_PATH}")

Конфигурация загружена
Данные: /content/drive/MyDrive/VKR/code/data/data_after_eda.csv


In [3]:
import os
import pandas as pd
import numpy as np
from collections import Counter
from tqdm.notebook import tqdm
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    pipeline
)
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings("ignore")

# Папка для промежуточных сохранений
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Checkpoints: {os.path.abspath(CHECKPOINT_DIR)}")
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Checkpoints: /content/checkpoints
GPU доступен: True
GPU: Tesla T4


In [4]:
# ============================================================
# КОНФИГУРАЦИЯ — меняй здесь
# ============================================================

DATA_PATH = "drive/MyDrive/VKR/code/Data/data.json"          # Путь к исходным данным
TARGET_COL = "label"                  # Колонка с метками классов
TEXT_COL = "text"                     # Колонка с текстами

SMALL_THRESHOLD = 25                  # Малые блоки: < 25 примеров
MEDIUM_THRESHOLD = 50                 # Средние блоки: 25–49 примеров
TARGET_COUNT = 50                     # До скольких примеров дополнять

# Модель для генерации
GENERATOR_MODEL = "ai-forever/rugpt3large_based_on_gpt2"

# Модель для фильтра перплексии
PERPLEXITY_MODEL = "ai-forever/rugpt3large_based_on_gpt2"
MAX_PERPLEXITY = 150.0

# Модель для эмбеддингов (дедупликация)
EMBED_MODEL = "cointegrated/rubert-tiny2"
SIMILARITY_THRESHOLD = 0.9

# Классификатор (укажи путь если есть дообученный)
CLASSIFIER_MODEL = None               # Например: "./my_classifier"
CLASSIFIER_THRESHOLD = 0.5

print("Конфигурация загружена")

Конфигурация загружена


## 1. Загрузка и анализ данных

Checkpoint: `checkpoints/01_blocks_to_augment.csv`

In [5]:
CKPT_BLOCKS = os.path.join(CHECKPOINT_DIR, "01_blocks_to_augment.csv")

# Загружаем исходные данные
data = pd.read_json(DATA_PATH)
print(f"Загружено {len(data)} строк, {data[TARGET_COL].nunique()} классов")
data.head()

Загружено 1774 строк, 36 классов


,idx,text,label
0,10026,[PERSON]\n\nУважаемый [PERSON]!\n\n[ORGANIZATI...,Блок директора по проектированию
1,1005,[ORGANIZATION] инжиниринг общество с ограничен...,Блок деректора по газу
2,1010,[ORGANIZATION] ИНВЕСТ Общество с ограниченной ...,Блок заместителя генерального директора по без...
3,10108,"[ORGANIZATION] ОГРН: [ID], ИНН/КПП: [ID] РФ, [...",Блок технического директора
4,1013,[ORGANIZATION] филиал [OBJECT] имени [PERSON] ...,Блок технического директора


In [6]:
class_counts = data[TARGET_COL].value_counts()

print("=" * 60)
print("РАСПРЕДЕЛЕНИЕ КЛАССОВ")
print("=" * 60)
print(f"Всего классов:  {len(class_counts)}")
print(f"Всего примеров: {len(data)}")
print(f"Медиана:        {class_counts.median():.0f}")
print(f"Минимум:        {class_counts.min()} ({class_counts.idxmin()})")
print(f"Максимум:       {class_counts.max()} ({class_counts.idxmax()})")
print()

small_blocks  = class_counts[class_counts < SMALL_THRESHOLD]
medium_blocks = class_counts[(class_counts >= SMALL_THRESHOLD) & (class_counts < MEDIUM_THRESHOLD)]
large_blocks  = class_counts[class_counts >= MEDIUM_THRESHOLD]

print(f"Малых блоков  (<{SMALL_THRESHOLD}):                {len(small_blocks)}")
print(f"Средних блоков ({SMALL_THRESHOLD}–{MEDIUM_THRESHOLD - 1}):            {len(medium_blocks)}")
print(f"Больших блоков (>={MEDIUM_THRESHOLD}):              {len(large_blocks)}")
print()

print("МАЛЫЕ БЛОКИ:")
print("-" * 40)
for lbl, cnt in small_blocks.items():
    print(f"  {lbl}: {cnt} примеров")

print()
print("СРЕДНИЕ БЛОКИ:")
print("-" * 40)
for lbl, cnt in medium_blocks.items():
    print(f"  {lbl}: {cnt} примеров")

# Объединяем блоки под аугментацию
blocks_to_augment = pd.concat([small_blocks, medium_blocks])

# Сохраняем checkpoint
blocks_to_augment.to_csv(CKPT_BLOCKS, header=True)
print(f"\nCheckpoint сохранён: {CKPT_BLOCKS}")

РАСПРЕДЕЛЕНИЕ КЛАССОВ
Всего классов:  36
Всего примеров: 1774
Медиана:        25
Минимум:        2 (Имущественные вопросы)
Максимум:       249 (Блок технического директора)

Малых блоков  (<25):                18
Средних блоков (25–49):            8
Больших блоков (>=50):              10

МАЛЫЕ БЛОКИ:
----------------------------------------
  Блок директора по портфелю: 24 примеров
  Управление землеустроительных работ: 23 примеров
  Проект "Трубопроводный транспорт Главного НГКМ": 20 примеров
  Проект "Восточный": 18 примеров
  Блок директора по персоналу: 18 примеров
  Проект «Обустройство объектов Новой нефти»: 18 примеров
  Блок заместителя генерального директора по имуществу: 17 примеров
  Проект "Южный": 12 примеров
  Блок бизнес-директора: 8 примеров
  Управление коммуникаций: 7 примеров
  Проект «Обустройство Интересного лицензионного участка»: 7 примеров
  Проект "Обустройство площадных объектов НГКМ Поменбше": 6 примеров
  Блок исполнительного директора по реализации проекта

## 2. Загрузка генеративной модели

In [7]:
print(f"Загружаем модель генерации: {GENERATOR_MODEL}")

gen_tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
gen_model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Модель загружена")

Загружаем модель генерации: ai-forever/rugpt3large_based_on_gpt2


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена


## 3. Генерация синтетических данных

Checkpoint: `checkpoints/02_generated.csv`  
Если файл уже существует — генерация пропускается.

In [8]:
def build_prompt(label: str, examples: list, example_text: str = "") -> str:
    """
    Короткий промпт для rugpt3large.
    
    Эта модель — не instruct, она просто продолжает текст.
    Поэтому промпт должен выглядеть как начало документа,
    а не как инструкция для ассистента.
    
    Берём только 1-2 примера чтобы уложиться в 2048 токенов.
    """
    # Берём максимум 2 коротких примера (экономим токены)
    selected = examples[:2]
    
    prompt = f'Категория писем: "{label}"\n\n'
    
    for i, ex in enumerate(selected, 1):
        # Обрезаем каждый пример до 300 символов чтобы не раздувать промпт
        short_ex = ex[:300].rsplit(' ', 1)[0]  # обрезаем по слову
        prompt += f"Пример {i}:\n{short_ex}\n\n"
    
    # Начало нового письма — модель просто продолжит текст
    prompt += "Новое письмо:\n"
    
    return prompt


def generate_one_example(model, tokenizer, prompt: str,
                         temperature: float = 0.9, top_p: float = 0.92) -> str:
    """
    Генерируем ОДИН пример за вызов.
    Так стабильнее — rugpt3large плохо держит формат на длинных генерациях.
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024  # половина контекста на промпт, половина на генерацию
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            repetition_penalty=1.3,     # чуть сильнее штраф за повторы
            no_repeat_ngram_size=4,     # запрет повтора 4-грамм
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    # Декодируем только новые токены (без промпта)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Обрезаем по первому явному концу письма
    # (модель может начать генерить следующее письмо или мусор)
    stop_markers = ["Пример ", "Категория писем:", "Новое письмо:", "\n\n\n"]
    for marker in stop_markers:
        if marker in generated:
            generated = generated[:generated.index(marker)].strip()

    return generated


def generate_examples(model, tokenizer, label: str, examples: list,
                      n_generate: int = 10) -> list:
    """
    Генерируем n_generate примеров для одного класса.
    Каждый пример — отдельный вызов модели.
    Медленнее, но надёжнее и разнообразнее.
    """
    prompt = build_prompt(label, examples)
    generated = []

    for i in range(n_generate):
        # Варьируем температуру для разнообразия
        temp = 0.8 + (i % 5) * 0.05  # от 0.8 до 1.0

        text = generate_one_example(model, tokenizer, prompt, temperature=temp)

        # Фильтруем слишком короткие и пустые
        if len(text) > 50:
            generated.append(text)

        # Прогресс
        if (i + 1) % 5 == 0:
            print(f"  [{label}] {i+1}/{n_generate} — получено {len(generated)} валидных")

    print(f"  Класс '{label}': сгенерировано {len(generated)}/{n_generate}")
    return generated


print("Функции генерации определены")

Функции генерации определены


In [9]:
# ============================================================
# ЭТАП 2: ГЕНЕРАЦИЯ СИНТЕТИЧЕСКИХ ДАННЫХ
# ============================================================

CKPT_GENERATED = os.path.join(CHECKPOINT_DIR, "02_generated.csv")

if os.path.exists(CKPT_GENERATED):
    print(f"Найден checkpoint: {CKPT_GENERATED}")
    generated_df = pd.read_csv(CKPT_GENERATED)
    print(f"Загружено {len(generated_df)} сгенерированных примеров — генерация пропускается")

else:
    # Загружаем список блоков если его нет в памяти
    if "blocks_to_augment" not in dir():
        blocks_to_augment = pd.read_csv(CKPT_BLOCKS, index_col=0).squeeze()
        print(f"blocks_to_augment загружен из checkpoint: {len(blocks_to_augment)} блоков")

    # Проверяем есть ли частичный checkpoint (если ядро падало посреди генерации)
    CKPT_PARTIAL = os.path.join(CHECKPOINT_DIR, "02_generated_partial.csv")
    if os.path.exists(CKPT_PARTIAL):
        partial_df = pd.read_csv(CKPT_PARTIAL)
        all_generated = partial_df.to_dict("records")
        done_labels = set(partial_df[TARGET_COL].unique())
        print(f"Найден частичный checkpoint: {len(all_generated)} примеров")
        print(f"Уже обработаны классы: {done_labels}")
    else:
        all_generated = []
        done_labels = set()

    print("=" * 60)
    print("ГЕНЕРАЦИЯ СИНТЕТИЧЕСКИХ ДАННЫХ")
    print("=" * 60)

    for label, count in tqdm(blocks_to_augment.items(), desc="Генерация"):
        # Пропускаем классы которые уже сгенерированы (после перезапуска ядра)
        if label in done_labels:
            print(f"  [{label}] уже сгенерирован — пропускаем")
            continue

        n_needed = TARGET_COUNT - count
        if n_needed <= 0:
            continue

        # Достаём оригинальные примеры этого класса
        original_examples = data[data[TARGET_COL] == label][TEXT_COL].tolist()

        # Генерируем по одному примеру за вызов
        generated = generate_examples(
            gen_model, gen_tokenizer, label, original_examples, n_generate=n_needed
        )

        for text in generated:
            all_generated.append({
                TEXT_COL: text,
                TARGET_COL: label,
                "is_synthetic": True
            })

        # Сохраняем частичный checkpoint после каждого класса
        # Если ядро упадёт — при перезапуске продолжим с этого места
        pd.DataFrame(all_generated).to_csv(
            CKPT_PARTIAL, index=False, encoding="utf-8-sig"
        )
        print(f"  Частичный checkpoint: {len(all_generated)} примеров сохранено")

    # Генерация завершена — сохраняем финальный checkpoint
    generated_df = pd.DataFrame(all_generated)
    generated_df.to_csv(CKPT_GENERATED, index=False, encoding="utf-8-sig")

    # Удаляем частичный checkpoint — он больше не нужен
    if os.path.exists(CKPT_PARTIAL):
        os.remove(CKPT_PARTIAL)

    print(f"\nВсего сгенерировано: {len(generated_df)} примеров")
    print(f"Финальный checkpoint: {CKPT_GENERATED}")

generated_df.head()

Найден частичный checkpoint: 115 примеров
Уже обработаны классы: {'Проект "Трубопроводный транспорт Главного НГКМ"', 'Управление землеустроительных работ', 'Блок директора по портфелю', 'Проект "Восточный"'}
ГЕНЕРАЦИЯ СИНТЕТИЧЕСКИХ ДАННЫХ


Генерация: 0it [00:00, ?it/s]

  [Блок директора по портфелю] уже сгенерирован — пропускаем
  [Управление землеустроительных работ] уже сгенерирован — пропускаем
  [Проект "Трубопроводный транспорт Главного НГКМ"] уже сгенерирован — пропускаем
  [Проект "Восточный"] уже сгенерирован — пропускаем
  [Блок директора по персоналу] 5/32 — получено 5 валидных
  [Блок директора по персоналу] 10/32 — получено 10 валидных
  [Блок директора по персоналу] 15/32 — получено 15 валидных
  [Блок директора по персоналу] 20/32 — получено 20 валидных
  [Блок директора по персоналу] 25/32 — получено 25 валидных
  [Блок директора по персоналу] 30/32 — получено 30 валидных
  Класс 'Блок директора по персоналу': сгенерировано 32/32
  Частичный checkpoint: 147 примеров сохранено
  [Проект «Обустройство объектов Новой нефти»] 5/32 — получено 5 валидных
  [Проект «Обустройство объектов Новой нефти»] 10/32 — получено 10 валидных
  [Проект «Обустройство объектов Новой нефти»] 15/32 — получено 15 валидных
  [Проект «Обустройство объектов Новой

,text,label,is_synthetic
0,"Здравствуйте, коллеги! Я очень рад вам сообщит...",Блок директора по портфелю,True
1,Отчет об изменении отчета о прибылях и убытках...,Блок директора по портфелю,True
2,"Организация, адрес - не указан ______. Количес...",Блок директора по портфелю,True
3,Отказ в проведении проверки компании Инвест Ко...,Блок директора по портфелю,True
4,"Письмо № 121/187208@192176, отправлено 18 авгу...",Блок директора по портфелю,True


In [14]:
generated_df['text'][0]

'Здравствуйте, коллеги! Я очень рад вам сообщить о поступлении в компанию нового сотрудника - Андрея Петрова, который к настоящему моменту является одним из ведущих специалистов компании и имеет большой опыт управления персоналом. Андрей уже давно работает в нашей организации и до настоящего момента успешно руководил такими проектами как развитие сети аптек ОАО "Магнит", работа над проектом реконструкции аптечного пункта в Москве, проект строительства торгово-развлекательного центра, а также ряд других проектов для всей страны \n http://www.marketingforums.ru/viewtopic.php?f=14&start=0#entry1090229 \xa0http://worldofinfocommerceweb2cms4data.com/index.html\nНастоящий волшебник не только делает волшебство Настоящее чудо происходит внутри человека или за его пределами Оригинальный сайт профессора Преображенского\nФундаментальные законы психического здоровья\n1.. Психические расстройства являются выражением внутренних конфликтов между человеком и окружающим миром. Причиняют страдания други

In [13]:
generated_df.to_csv(f"drive/MyDrive/VKR/code/saves/data_after_generate.csv")

## 4. Фильтр 1 — Перплексия

Checkpoint: `checkpoints/03_after_perplexity.csv`

In [15]:
CKPT_PERPLEXITY = os.path.join(CHECKPOINT_DIR, "03_after_perplexity.csv")

if os.path.exists(CKPT_PERPLEXITY):
    print(f"Найден checkpoint: {CKPT_PERPLEXITY}")
    after_perplexity_df = pd.read_csv(CKPT_PERPLEXITY)
    print(f"Загружено {len(after_perplexity_df)} примеров после фильтра перплексии — шаг пропускается")
else:
    if "generated_df" not in dir():
        generated_df = pd.read_csv(CKPT_GENERATED)
        print(f"generated_df загружен из checkpoint: {len(generated_df)} примеров")

    print("--- Фильтр 1: Перплексия ---")
    print(f"Загружаем модель: {PERPLEXITY_MODEL}")

    ppl_tokenizer = AutoTokenizer.from_pretrained(PERPLEXITY_MODEL)
    ppl_model = AutoModelForCausalLM.from_pretrained(
        PERPLEXITY_MODEL, torch_dtype=torch.float16, device_map="auto"
    )
    ppl_model.eval()

    texts = generated_df[TEXT_COL].tolist()
    passed = []

    for text in tqdm(texts, desc="Перплексия"):
        inputs = ppl_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(ppl_model.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = ppl_model(**inputs, labels=inputs["input_ids"])
            perplexity = torch.exp(outputs.loss).item()
        passed.append(perplexity <= MAX_PERPLEXITY)

    after_perplexity_df = generated_df[passed].reset_index(drop=True)

    n_passed = sum(passed)
    print(f"Прошло: {n_passed}/{len(texts)} ({n_passed/len(texts)*100:.1f}%)")

    after_perplexity_df.to_csv(CKPT_PERPLEXITY, index=False, encoding="utf-8-sig")
    print(f"Checkpoint сохранён: {CKPT_PERPLEXITY}")

    # Освобождаем память
    del ppl_model
    torch.cuda.empty_cache()

print(f"После фильтра перплексии: {len(after_perplexity_df)} примеров")

--- Фильтр 1: Перплексия ---
Загружаем модель: ai-forever/rugpt3large_based_on_gpt2


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Перплексия:   0%|          | 0/824 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Прошло: 824/824 (100.0%)
Checkpoint сохранён: checkpoints/03_after_perplexity.csv
После фильтра перплексии: 824 примеров


## 5. Фильтр 2 — Дедупликация

Checkpoint: `checkpoints/04_after_dedup.csv`

In [16]:
CKPT_DEDUP = os.path.join(CHECKPOINT_DIR, "04_after_dedup.csv")

if os.path.exists(CKPT_DEDUP):
    print(f"Найден checkpoint: {CKPT_DEDUP}")
    after_dedup_df = pd.read_csv(CKPT_DEDUP)
    print(f"Загружено {len(after_dedup_df)} примеров после дедупликации — шаг пропускается")
else:
    if "after_perplexity_df" not in dir():
        after_perplexity_df = pd.read_csv(CKPT_PERPLEXITY)
        print(f"after_perplexity_df загружен из checkpoint: {len(after_perplexity_df)} примеров")

    print("--- Фильтр 2: Дедупликация ---")
    embed_model = SentenceTransformer(EMBED_MODEL)

    dedup_mask_ordered = []

    for label in after_perplexity_df[TARGET_COL].unique():
        gen_mask = after_perplexity_df[TARGET_COL] == label
        gen_texts = after_perplexity_df[gen_mask][TEXT_COL].tolist()
        orig_texts = data[data[TARGET_COL] == label][TEXT_COL].tolist()

        gen_embeddings  = embed_model.encode(gen_texts, show_progress_bar=False)
        orig_embeddings = embed_model.encode(orig_texts, show_progress_bar=False)

        class_passed = []
        for i, gen_emb in enumerate(gen_embeddings):
            gen_emb = gen_emb.reshape(1, -1)
            sim_orig = cosine_similarity(gen_emb, orig_embeddings).max()
            sim_gen  = cosine_similarity(gen_emb, gen_embeddings[:i]).max() if i > 0 else 0
            class_passed.append(
                sim_orig < SIMILARITY_THRESHOLD and sim_gen < SIMILARITY_THRESHOLD
            )

        dedup_mask_ordered.extend(class_passed)
        n = sum(class_passed)
        print(f"  {label}: {n}/{len(gen_texts)} уникальных")

    after_dedup_df = after_perplexity_df[dedup_mask_ordered].reset_index(drop=True)

    n_passed = len(after_dedup_df)
    n_total  = len(after_perplexity_df)
    print(f"\nПрошло: {n_passed}/{n_total} ({n_passed/n_total*100:.1f}%)")

    after_dedup_df.to_csv(CKPT_DEDUP, index=False, encoding="utf-8-sig")
    print(f"Checkpoint сохранён: {CKPT_DEDUP}")

print(f"После дедупликации: {len(after_dedup_df)} примеров")

--- Фильтр 2: Дедупликация ---


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Блок директора по портфелю: 26/26 уникальных
  Управление землеустроительных работ: 25/27 уникальных
  Проект "Трубопроводный транспорт Главного НГКМ": 30/30 уникальных
  Проект "Восточный": 32/32 уникальных
  Блок директора по персоналу: 32/32 уникальных
  Проект «Обустройство объектов Новой нефти»: 31/31 уникальных
  Блок заместителя генерального директора по имуществу: 33/33 уникальных
  Проект "Южный": 38/38 уникальных
  Блок бизнес-директора: 42/42 уникальных
  Управление коммуникаций: 43/43 уникальных
  Проект «Обустройство Интересного лицензионного участка»: 43/43 уникальных
  Проект "Обустройство площадных объектов НГКМ Поменбше": 44/44 уникальных
  Блок исполнительного директора по реализации проекта "Большое месторождение": 45/45 уникальных
  Проект «Обустройство объектов Новейшей нейти»: 46/47 уникальных
  Имущественные вопросы: 46/48 уникальных
  Подразделение по информационным технологиям: 48/48 уникальных
  Проект «Трубопроводный транспорт Ещё одного НГКМ»: 48/48 уникал

## 6. Фильтр 3 — Классификатор / LLM-арбитр

Checkpoint: `checkpoints/05_after_classifier.csv`

- Если `CLASSIFIER_MODEL = None` — для блоков ≤10 примеров используется LLM-арбитр, остальные пропускаются
- Если указан путь к классификатору — используется он

In [ ]:
CKPT_CLASSIFIER = os.path.join(CHECKPOINT_DIR, "05_after_classifier.csv")

if os.path.exists(CKPT_CLASSIFIER):
    print(f"Найден checkpoint: {CKPT_CLASSIFIER}")
    validated_df = pd.read_csv(CKPT_CLASSIFIER)
    print(f"Загружено {len(validated_df)} примеров после классификатора — шаг пропускается")
else:
    if "after_dedup_df" not in dir():
        after_dedup_df = pd.read_csv(CKPT_DEDUP)
        print(f"after_dedup_df загружен из checkpoint: {len(after_dedup_df)} примеров")

    if "blocks_to_augment" not in dir():
        blocks_to_augment = pd.read_csv(CKPT_BLOCKS, index_col=0).squeeze()

    all_labels = data[TARGET_COL].unique().tolist()
    clf_mask = []

    # Загружаем классификатор или генеративную модель для арбитра
    if CLASSIFIER_MODEL:
        print(f"--- Фильтр 3: Классификатор ({CLASSIFIER_MODEL}) ---")
        clf_pipeline = pipeline(
            "text-classification",
            model=CLASSIFIER_MODEL,
            tokenizer=CLASSIFIER_MODEL,
            device=0 if torch.cuda.is_available() else -1,
            top_k=None
        )
    else:
        print("--- Фильтр 3: LLM-арбитр (классификатор не задан) ---")
        # Переиспользуем gen_model если он ещё в памяти
        if "gen_model" not in dir():
            gen_tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
            gen_model = AutoModelForCausalLM.from_pretrained(
                GENERATOR_MODEL, torch_dtype=torch.float16, device_map="auto"
            )
        labels_str = "\n".join([f"- {lbl}" for lbl in all_labels])

    for label in after_dedup_df[TARGET_COL].unique():
        gen_mask  = after_dedup_df[TARGET_COL] == label
        gen_texts = after_dedup_df[gen_mask][TEXT_COL].tolist()
        original_count = blocks_to_augment.get(label, 0)

        if CLASSIFIER_MODEL:
            class_filter = []
            for text in tqdm(gen_texts, desc=f"Классификатор [{label}]", leave=False):
                predictions = clf_pipeline(text, truncation=True, max_length=512)
                target_prob = next(
                    (p["score"] for p in predictions[0] if p["label"] == label), 0.0
                )
                class_filter.append(target_prob >= CLASSIFIER_THRESHOLD)

        elif original_count <= 10:
            # LLM-арбитр для очень малых блоков
            class_filter = []
            for text in tqdm(gen_texts, desc=f"LLM-арбитр [{label}]", leave=False):
                prompt = f"""Определи категорию следующего делового письма.

Письмо:
{text}

Возможные категории:
{labels_str}

Ответь ТОЛЬКО названием одной категории, без пояснений:"""
                inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
                with torch.no_grad():
                    outputs = gen_model.generate(**inputs, max_new_tokens=50, temperature=0.1, do_sample=True)
                response = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)
                response = response[len(prompt):].strip()
                class_filter.append(label.lower() in response.lower())
        else:
            # Нет ни классификатора, ни LLM-арбитра — пропускаем все
            class_filter = [True] * len(gen_texts)

        n = sum(class_filter)
        print(f"  {label}: {n}/{len(gen_texts)} прошло")
        clf_mask.extend(class_filter)

    validated_df = after_dedup_df[clf_mask].reset_index(drop=True)

    n_passed = len(validated_df)
    n_total  = len(after_dedup_df)
    print(f"\nПрошло: {n_passed}/{n_total} ({n_passed/n_total*100:.1f}%)")

    validated_df.to_csv(CKPT_CLASSIFIER, index=False, encoding="utf-8-sig")
    print(f"Checkpoint сохранён: {CKPT_CLASSIFIER}")

print(f"После каскада валидации: {len(validated_df)} примеров")

--- Фильтр 3: LLM-арбитр (классификатор не задан) ---
  Блок директора по портфелю: 26/26 прошло
  Управление землеустроительных работ: 25/25 прошло
  Проект "Трубопроводный транспорт Главного НГКМ": 30/30 прошло
  Проект "Восточный": 32/32 прошло
  Блок директора по персоналу: 32/32 прошло
  Проект «Обустройство объектов Новой нефти»: 31/31 прошло
  Блок заместителя генерального директора по имуществу: 33/33 прошло
  Проект "Южный": 38/38 прошло


LLM-арбитр [Блок бизнес-директора]:   0%|          | 0/42 [00:00<?, ?it/s]

  Блок бизнес-директора: 0/42 прошло


LLM-арбитр [Управление коммуникаций]:   0%|          | 0/43 [00:00<?, ?it/s]

  Управление коммуникаций: 0/43 прошло


LLM-арбитр [Проект «Обустройство Интересного лицензионного участка»]:   0%|          | 0/43 [00:00<?, ?it/s]

## 7. Итоги валидации

In [ ]:
if "generated_df" not in dir():
    generated_df = pd.read_csv(CKPT_GENERATED)
if "validated_df" not in dir():
    validated_df = pd.read_csv(CKPT_CLASSIFIER)

initial_count = len(generated_df)
final_count   = len(validated_df)

print("=" * 60)
print("ИТОГИ ВАЛИДАЦИИ")
print("=" * 60)
print(f"Было сгенерировано: {initial_count}")
print(f"Прошло валидацию:   {final_count}")
print(f"Отсеяно:            {initial_count - final_count}")
print(f"Доля прошедших:     {final_count/initial_count*100:.1f}%")
print()
print("По классам:")
for lbl in validated_df[TARGET_COL].unique():
    cnt = (validated_df[TARGET_COL] == lbl).sum()
    print(f"  {lbl}: {cnt} примеров")

## 8. Сборка финального датасета

Результат: `augmented_dataset.csv`

In [ ]:
if "data" not in dir():
    data = pd.read_csv(DATA_PATH)
if "validated_df" not in dir():
    validated_df = pd.read_csv(CKPT_CLASSIFIER)

original_data = data.copy()
original_data["is_synthetic"] = False

final_data = pd.concat([original_data, validated_df], ignore_index=True)
final_data = final_data.sample(frac=1, random_state=42).reset_index(drop=True)

print("=" * 60)
print("ФИНАЛЬНЫЙ ДАТАСЕТ")
print("=" * 60)
print(f"Оригинальных:  {(~final_data['is_synthetic']).sum()}")
print(f"Синтетических: {final_data['is_synthetic'].sum()}")
print(f"Всего:         {len(final_data)}")
print()
print("Распределение классов после аугментации:")
counts = final_data[TARGET_COL].value_counts()
for lbl, cnt in counts.items():
    synthetic = ((final_data[TARGET_COL] == lbl) & final_data["is_synthetic"]).sum()
    print(f"  {lbl}: {cnt} (синтетических: {synthetic})")

final_data.to_csv("augmented_dataset.csv", index=False, encoding="utf-8-sig")
print("\nДатасет сохранён: augmented_dataset.csv")

## 9. Выборка для ручной валидации

Результат: `manual_validation_sample.csv`

In [ ]:
N_PER_CLASS = 10  # Сколько примеров на класс брать для проверки

if "validated_df" not in dir():
    validated_df = pd.read_csv(CKPT_CLASSIFIER)

samples = []
for lbl in validated_df[TARGET_COL].unique():
    class_data = validated_df[validated_df[TARGET_COL] == lbl]
    n_sample = min(N_PER_CLASS, len(class_data))
    samples.append(class_data.sample(n=n_sample, random_state=42))

sample_df = pd.concat(samples, ignore_index=True)

# Колонки для разметки
sample_df["correct_class"] = ""   # Соответствует классу? (1/0)
sample_df["natural_text"]  = ""   # Естественный текст? (1/0)
sample_df["is_copy"]       = ""   # Копия оригинала? (1/0)
sample_df["comment"]       = ""   # Свободный комментарий

sample_df.to_csv("manual_validation_sample.csv", index=False, encoding="utf-8-sig")
print(f"Сохранено {len(sample_df)} примеров в manual_validation_sample.csv")
print("Заполни колонки: correct_class, natural_text, is_copy, comment")

sample_df.head()